In [1]:
from pyspark.sql import SparkSession

# Initialise Spark session
spark = (
    SparkSession.builder.appName("Rideshare_Analysis")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

your 131072x1 screen size is bogus. expect trouble
24/08/23 21:07:55 WARN Utils: Your hostname, LAPTOP-354CCOC2 resolves to a loopback address: 127.0.1.1; using 172.21.14.166 instead (on interface eth0)
24/08/23 21:07:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/23 21:07:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Load FHVHV data
fhvhv_df = spark.read.parquet("../data/raw")

# Load weather data
weather_df = spark.read.csv('../data/raw_csv/NYC hourly weather 2023-05-01 to 2024-05-01.csv', header=True, inferSchema=True)

# Load public holidays data
holidays_2023_df = spark.read.csv('../data/raw_csv/2023_holidays.csv', header=True, inferSchema=True)
holidays_2024_df = spark.read.csv('../data/raw_csv/2024_holidays.csv', header=True, inferSchema=True)

# Combine 2023 and 2024 holidays
holidays_df = holidays_2023_df.union(holidays_2024_df)


In [16]:
# Examine the initial shape of all the datasets
from 

get_dataset_shape(fhvhv_df)
get_dataset_shape(weather_df)
get_dataset_shape(holidays_df)

ImportError: cannot import name 'tlc' from 'scripts' (unknown location)

In [39]:
import os
print(os.getcwd())


/mnt/c/Users/Ryan/Documents/Refactored_Py_DS_ML_Bootcamp-master/18-Principal-Component-Analysis/project-1-individual-alistaircwh/notebook


In [15]:
# Display schemas for all datasets
fhvhv_df.printSchema()
weather_df.printSchema()
holidays_df.printSchema()

# Preview a few records
fhvhv_df.show(7)
weather_df.show(7)
holidays_df.show(5)

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [5]:
# Count missing values in each column and display as a table
from pyspark.sql.functions import col, sum

def get_missing_value_counts(dataset):

    # Initialise an empty list to hold the sum columns
    sum_columns = []

    # Iterate over each column, summing their counts of NULL values
    for c in dataset.columns:
        sum_expression = sum(col(c).isNull().cast("int")).alias(c + '_count')
        sum_columns.append(sum_expression)

    # Selection and display
    missing_values = dataset.select(sum_columns)
    missing_values.show()

    return


+-----------------------+--------------------------+--------------------------+----------------------+-----------------------+---------------------+----------------------+------------------+------------------+----------------+---------------+-------------------------+-----------+---------+---------------+--------------------------+-----------------+----------+----------------+-------------------------+-----------------------+------------------------+----------------------+--------------------+
|hvfhs_license_num_count|dispatching_base_num_count|originating_base_num_count|request_datetime_count|on_scene_datetime_count|pickup_datetime_count|dropoff_datetime_count|PULocationID_count|DOLocationID_count|trip_miles_count|trip_time_count|base_passenger_fare_count|tolls_count|bcf_count|sales_tax_count|congestion_surcharge_count|airport_fee_count|tips_count|driver_pay_count|shared_request_flag_count|shared_match_flag_count|access_a_ride_flag_count|wav_request_flag_count|wav_match_flag_count|
+-

In [10]:
# Summary statistics
sample_df = fhvhv_df.sample(fraction=0.10, seed=42)  # 10% of the data
sample_df.describe().show()


+-------+-----------------+--------------------+--------------------+------------------+------------------+-----------------+------------------+-------------------+------------------+------------------+-----------------+--------------------+-------------------+------------------+-----------------+-------------------+-----------------+------------------+----------------+--------------+
|summary|hvfhs_license_num|dispatching_base_num|originating_base_num|      PULocationID|      DOLocationID|       trip_miles|         trip_time|base_passenger_fare|             tolls|               bcf|        sales_tax|congestion_surcharge|        airport_fee|              tips|       driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-------+-----------------+--------------------+--------------------+------------------+------------------+-----------------+------------------+-------------------+------------------+------------------+-----------------+----

In [ ]:

# Convert the date column to a proper datetime format (assuming the column is named 'date')
combined_holidays_df = holidays_df.withColumn('date', to_date(holidays_df['date'], 'yyyy-MM-dd'))